# MuseTalk GPU Acceptance Pilot — Swar Avatar Engine

**Purpose:** run the *first real* MuseTalk inference for the Mohan Sir digital-twin pilot and produce acceptance evidence (GPU telemetry → inference → Bunny upload → read-back → screenshots).

**Replaces:** the retired mux-only "acceptance" scripts. Nothing in this notebook loops or muxes media as a substitute for inference — the adapter fails closed if inference produces no output.

**Approved pilot scope** (docs/MUSETALK_GPU_ACCEPTANCE_REPORT.md):
- ✅ YuNet face detector (MIT) **instead of S3FD** (no license identifier)
- ✅ Landmark-derived geometric mask **instead of BiSeNet** face-parse (provenance unclear)
- ✅ Weights pinned: MuseTalk HF revision `2bcb936e…`, `unet.pth` verified against SHA-256 `7ebf6c98…`
- ❌ SyncNet, S3FD, BiSeNet, resnet18 face-parse: **never downloaded**

**Safety rules:**
1. Secrets ONLY via the Colab **Secrets** panel (🔑). Never paste tokens into cells, never print them.
2. Use only **approved, consented** source video + audio (10 s pilot pair).
3. A completed run is **evidence for review**, not automatic acceptance — visual QA (identity / lip-sync / flicker) is a human step.
4. Runtime: GPU runtime required (Runtime → Change runtime type → T4 or better).

In [ ]:
# 1) GPU check — the pilot requires an authorized NVIDIA GPU session
!nvidia-smi
import torch, sys
assert torch.cuda.is_available(), "No CUDA GPU. Runtime -> Change runtime type -> GPU."
print(f"\nPyTorch {torch.__version__} | GPU: {torch.cuda.get_device_name(0)} | Python {sys.version.split()[0]}")

In [ ]:
# 2) Fetch pilot scripts (sparse clone of the project repo) + MuseTalk clone + inference-path deps
import os

if not os.path.isdir("/content/swaryoga"):
    !git clone --filter=blob:none --sparse https://github.com/globalswaryoga-ai/swaryoga.com-db.git /content/swaryoga
%cd /content/swaryoga
!git sparse-checkout set scripts docs colab

PILOT_DIR = "/content/swaryoga/scripts/musetalk_pilot"
ADAPTER = "/content/swaryoga/scripts/avatar-gpu-adapter-server.py"
assert os.path.isfile(ADAPTER), "Adapter script missing after sparse checkout"
assert os.path.isfile(os.path.join(PILOT_DIR, "compat_stubs.py")), "compat_stubs.py missing — sparse checkout incomplete"

if not os.path.isdir("/content/MuseTalk"):
    !git clone https://github.com/TMElyralab/MuseTalk.git /content/MuseTalk

%cd /content/MuseTalk

# ---- Inference-path dependencies only -------------------------------------
# Deliberate omissions (grep-verified against MuseTalk + mmpose import chains):
#   * tensorflow / tensorboard / gradio — upstream extras, never imported
#   * mim / source builds               — mim falls back to sdist on py3.12 and hangs
#   * mmdet (pip)                       — mmdet 3.1.0 asserts mmcv<2.1.0, but openmmlab
#                                         has no mmcv wheel for py>=3.10 below 2.1.0;
#                                         only rtmo_head imports it, never on this path
#   * xtcocotools (pip)                 — PyPI has no wheels past cp39 and no usable
#                                         sdist; compat_stubs delegates to pycocotools
# numpy<2 + setuptools<81 keep mmengine/mmpose importable (pkg_resources).
!pip install -q "numpy<2" "setuptools<81"

# mmcv: mmcv-lite is a PURE WHEEL with no torch-ABI coupling — works with whatever
# torch Colab ships (2.4 → 2.6+); no torch downgrade ever needed. MuseTalk itself
# imports zero mmcv.ops; the MultiScaleDeformableAttention warning is expected.
!pip install -q mmengine "mmcv-lite==2.2.0"

# mmpose: every 1.x release's PyPI metadata lists dead `chumpy` (unbuildable on
# modern pip) — install --no-deps and add only the eager-import deps:
#   munkres (codecs), json-tricks/scipy/pillow (dataset utils), pycocotools
#   (compat_stubs maps xtcocotools -> pycocotools).
# mmpose==1.3.2: only release verified import-clean on Python 3.12.
!pip install -q --no-deps "mmpose==1.3.2"
!pip install -q munkres json-tricks scipy pillow pycocotools

# Remaining MuseTalk runtime deps (pinned where upstream pins them).
!pip install -q "diffusers==0.30.2" "accelerate==0.28.0" opencv-python soundfile "transformers==4.39.2" "huggingface_hub[cli]" librosa einops omegaconf moviepy "imageio[ffmpeg]" gdown requests

# Fail fast: compat shims + the whole critical import chain BEFORE pulling 3.4 GB of weights.
import sys
sys.path.insert(0, PILOT_DIR)
import compat_stubs
compat_stubs.install()
from mmpose.apis import inference_topdown, init_model  # noqa: F401
from mmpose.structures import merge_data_samples  # noqa: F401
import torchvision  # noqa: F401
import mmcv, mmengine, mmpose, torch
print(f"OK | torch {torch.__version__} | mmcv-lite {mmcv.__version__} | "
      f"mmengine {mmengine.__version__} | mmpose {mmpose.__version__} | "
      f"torchvision {torchvision.__version__}")

%cd /content/swaryoga
print("Setup complete.")


In [ ]:
# 3) Download ONLY pilot-approved weights; record SHA-256 for every file
#    (S3FD / BiSeNet / resnet18 / SyncNet are excluded by design)
import os
os.environ["MUSETALK_HOME"] = "/content/MuseTalk"

!python /content/swaryoga/scripts/musetalk_pilot/download_pilot_weights.py --models-dir /content/MuseTalk/models

import json
manifest = json.load(open("/content/MuseTalk/models/pilot_weights_manifest.json"))
assert manifest["status"] == "complete", f"Weight download incomplete: {manifest['failures']}"
print(f"\nManifest: {len(manifest['files'])} files hashed @ revision {manifest['hf_revision_pinned'][:12]}…")

In [ ]:
# 4) Load secrets from the Colab Secrets panel — values are never displayed
from google.colab import user_data
import os

def require_secret(name: str) -> str:
    value = user_data.get(name)
    if not value:
        raise RuntimeError(
            f"Add Colab secret '{name}' (🔑 panel on the left). "
            "Never paste secrets into code cells."
        )
    return value

os.environ["BUNNY_STORAGE_ZONE_NAME"] = require_secret("BUNNY_STORAGE_ZONE_NAME")
os.environ["BUNNY_STORAGE_API_KEY"] = require_secret("BUNNY_STORAGE_API_KEY")
os.environ["BUNNY_STORAGE_CDN_HOST"] = require_secret("BUNNY_STORAGE_CDN_HOST")
os.environ.setdefault("BUNNY_STORAGE_REGION", "de")
os.environ["AVATAR_INFERENCE_TOKEN"] = require_secret("AVATAR_INFERENCE_TOKEN")
os.environ["MUSETALK_HOME"] = "/content/MuseTalk"

print("Secrets loaded into process env (values not shown).")

In [ ]:
# 5) Start the GPU adapter + Cloudflare quick tunnel (HTTPS — the worker contract
#    requires AVATAR_INFERENCE_URL to start with https://)
import os, re, subprocess, sys, time

# restart cleanly if the cell is re-run
subprocess.run(["pkill", "-f", "avatar-gpu-adapter-server"], capture_output=True)
subprocess.run(["pkill", "-f", "cloudflared tunnel"], capture_output=True)
time.sleep(1)

adapter = subprocess.Popen(
    [sys.executable, ADAPTER, "--port", "8080"],
    env=os.environ.copy(),
    stdout=open("/content/adapter.log", "w"),
    stderr=subprocess.STDOUT,
)

if not os.path.isfile("/usr/local/bin/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8080", "--no-autoupdate"],
    stdout=open("/content/cloudflared.log", "w"),
    stderr=subprocess.STDOUT,
)

tunnel_url = None
for _ in range(90):
    try:
        with open("/content/cloudflared.log") as fh:
            match = re.search(r"(https://[a-z0-9-]+\.trycloudflare\.com)", fh.read())
        if match:
            tunnel_url = match.group(1)
            break
    except FileNotFoundError:
        pass
    time.sleep(1)

assert tunnel_url, "cloudflared never printed a URL — inspect /content/cloudflared.log"
os.environ["AVATAR_INFERENCE_URL"] = tunnel_url + "/render"
print("AVATAR_INFERENCE_URL =", os.environ["AVATAR_INFERENCE_URL"])
print("(ephemeral per run — for the persistent worker, use a named tunnel or the GPU host's own HTTPS)")

In [ ]:
# 6) Adapter health — must report ready with all pilot weights present
import json, time, urllib.request

health = None
for _ in range(30):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8080/health", timeout=5) as resp:
            health = json.load(resp)
        break
    except Exception:
        time.sleep(1)

assert health, "Adapter /health never became reachable"
print(json.dumps({k: health[k] for k in ("status", "not_ready_reasons", "rendering", "gpu")}, indent=2))
print("Models:")
for name, info in health["models"]["required"].items():
    print(f"  {name:14s} present={info['present']} bytes={info['bytes']}")
assert health["status"] == "ready", f"Adapter not ready: {health['not_ready_reasons']}"

# also verify the HTTPS path end-to-end
with urllib.request.urlopen(os.environ["AVATAR_INFERENCE_URL"].replace("/render", "/health"), timeout=15) as resp:
    assert json.load(resp)["status"] == "ready"
print("HTTPS tunnel → adapter health: OK")

In [ ]:
# 7) RUN THE PILOT — 10 s approved source video + matching audio → MuseTalk → Bunny
#    Fill these with the approved, consented pilot pair (HTTPS URLs on Bunny CDN).
# Verified live (HTTP 200) + ffprobe-checked candidates from this repo's public assets:
#   video: twin_1 = 10.000 s, h264+aac, face clear/front-facing at t=1s and t=5s
#   audio: mohan_voice_..._4zsq5.mp3 = 9.792 s (voice-clone Mohan speech)
# Swap only with another consented pair if the reviewer prefers; both must stay HTTPS.
SOURCE_VIDEO_URL = "https://swaryoga.com/avatar-twins/twin_1_video.mp4"
AUDIO_URL        = "https://swaryoga.com/voice-clones/generated/mohan_voice_1794091814116_4zsq5.mp3"
JOB_ID           = "musetalk-pilot-10s"
# Fallbacks (also verified live): twin_2_video.mp4 (10.0 s) with the same audio;
# 30 s options: avatar-studio/videos/mohan_real_school_teacher_30s.mp4 +
# voice-clones/mohan-sir/mohan_voice_sample_02.mp3 (26.3 s).

assert SOURCE_VIDEO_URL.startswith("https://"), "source must be an HTTPS URL"
assert AUDIO_URL.startswith("https://"), "audio must be an HTTPS URL"
assert not SOURCE_VIDEO_URL.startswith("PASTE_"), "Set SOURCE_VIDEO_URL first"
assert not AUDIO_URL.startswith("PASTE_"), "Set AUDIO_URL first"

import json, os, subprocess, time, urllib.request

# GPU telemetry before
before = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,utilization.gpu", "--format=csv,noheader"],
                        capture_output=True, text=True).stdout.strip()
print("GPU before:", before)

payload = {
    "job_id": JOB_ID,
    "avatar_profile_id": "pilot-acceptance",
    "source_image_url": SOURCE_VIDEO_URL,
    "source_video_url": SOURCE_VIDEO_URL,
    "audio_url": AUDIO_URL,
    "script": "MuseTalk pilot acceptance run",
    "voice_id": "pilot",
    "language": "hi",
    "speech_speed": 1.0,
    "speech_pitch_hz": 0,
    "background_id": "source",
    "clothing_id": "source",
    "posture": "sitting",
    "output_requirement": {"format": "mp4", "storage": "Bunny CDN", "https_required": True},
}

request = urllib.request.Request(
    os.environ["AVATAR_INFERENCE_URL"],
    data=json.dumps(payload).encode("utf-8"),
    method="POST",
    headers={
        "Authorization": f"Bearer {os.environ['AVATAR_INFERENCE_TOKEN']}",
        "Content-Type": "application/json",
    },
)

started = time.time()
with urllib.request.urlopen(request, timeout=7200) as resp:
    result = json.load(resp)
elapsed = time.time() - started

after = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,utilization.gpu", "--format=csv,noheader"],
                       capture_output=True, text=True).stdout.strip()
print("GPU after: ", after)
print(f"Elapsed: {elapsed:.1f}s")
print(json.dumps(result, indent=2))

assert result.get("video_url", "").startswith("https://"), "Adapter did not return an HTTPS Bunny URL"
assert result.get("duration_seconds", 0) > 0, "Zero-duration output"
PILOT_RESULT = result

In [ ]:
# 8) Independent acceptance evidence: CDN read-back, ffprobe, screenshots, evidence JSON
import json, os, subprocess, urllib.request

video_url = PILOT_RESULT["video_url"]

# --- independent read-back over the public CDN (Range probe) ---
req = urllib.request.Request(video_url, headers={"Range": "bytes=0-0"})
with urllib.request.urlopen(req, timeout=60) as resp:
    cdn_status = resp.status
    resp.read(64)
assert cdn_status in (200, 206), f"CDN read-back failed: HTTP {cdn_status}"
print(f"CDN read-back: HTTP {cdn_status} OK")

# --- download for local verification ---
local = "/content/pilot_output.mp4"
urllib.request.urlretrieve(video_url, local)
size = os.path.getsize(local)

def probe(path):
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries",
         "format=duration,size:stream=codec_name,width,height,avg_frame_rate,nb_frames",
         "-of", "json", path], capture_output=True, text=True, check=True)
    return json.loads(out.stdout)

meta = probe(local)
print(json.dumps(meta, indent=2))
duration = float(meta["format"]["duration"])
assert duration > 0, "Output has zero duration"
assert abs(duration - PILOT_RESULT["duration_seconds"]) < 1.5, "Duration mismatch vs adapter response"

# --- screenshot frames for human visual QA ---
os.makedirs("/content/frames", exist_ok=True)
for label, t in (("start", duration * 0.15), ("mid", duration * 0.5), ("end", duration * 0.85)):
    subprocess.run([
        "ffmpeg", "-y", "-v", "error", "-ss", str(t), "-i", local,
        "-frames:v", "1", f"/content/frames/{label}.png",
    ], check=True)

from IPython.display import Image, display
for label in ("start", "mid", "end"):
    print(f"Frame @ {label}:")
    display(Image(filename=f"/content/frames/{label}.png"))

# --- GPU telemetry captured earlier; adapter log tail for inference proof ---
adapter_log = open("/content/adapter.log").read()
assert "musetalk-pilot-inference" in adapter_log, "Adapter log shows no inference invocation"
assert "Render complete" in adapter_log, "Adapter log shows no completed render"

weights_manifest = json.load(open("/content/MuseTalk/models/pilot_weights_manifest.json"))
evidence = {
    "pilot": "MuseTalk GPU Acceptance Pilot",
    "engine": result.get("engine"),
    "job_id": result.get("job_id"),
    "video_url": video_url,
    "adapter_duration_seconds": PILOT_RESULT["duration_seconds"],
    "ffprobe_duration_seconds": duration,
    "local_bytes": size,
    "cdn_read_back_http": cdn_status,
    "storage_read_back": result.get("read_back"),
    "elapsed_seconds": elapsed,
    "gpu_before": before,
    "gpu_after": after,
    "weights_manifest": weights_manifest,
    "replacements": {"face_detector": "YuNet (MIT)", "face_mask": "landmark-derived geometric mask"},
    "excluded": weights_manifest.get("excluded_artifacts"),
    "human_visual_qa": "PENDING — identity / lip-sync / flicker review required",
}
evidence_path = "/content/MUSETALK_PILOT_EVIDENCE.json"
with open(evidence_path, "w") as fh:
    json.dump(evidence, fh, indent=2)
print(f"\nEvidence written: {evidence_path} — download it and attach to the acceptance report.")

## Acceptance verdict checklist

A run only counts when **all** are true:

| # | Check | Where |
|---|-------|-------|
| 1 | Real GPU inference ran (not a mux) | adapter log contains `musetalk-pilot-inference` + `Render complete` |
| 2 | Weights pinned & hashed | `pilot_weights_manifest.json` — `unet.pth` == `7ebf6c98…` |
| 3 | Excluded assets absent | S3FD / BiSeNet / SyncNet never downloaded |
| 4 | Non-zero output duration | ffprobe + adapter agree |
| 5 | Bunny upload **and** read-back | adapter `read_back` + independent CDN Range probe |
| 6 | Screenshots pass human QA | identity, lip-sync vs audio, no flicker/seams at jaw mask edge |
| 7 | Evidence archived | `MUSETALK_PILOT_EVIDENCE.json` attached to `docs/MUSETALK_GPU_ACCEPTANCE_REPORT.md` |

**Until every row passes and the report status is updated from FAIL/BLOCKED, the pilot is not accepted.**

Stop the runtime when done (pkill adapter + cloudflared) to release the GPU.